In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import yaml
CONFIG_PATH = "../config.yaml"

In [2]:
gold = pl.read_parquet("../data/raw/Gold_1d.parquet")
nifty = pl.read_parquet("../data/raw/Nifty50_1d.parquet")
usdinr = pl.read_parquet("../data/raw/USDINR_1d.parquet")

In [3]:
nifty = nifty.rename({"('Close', '^NSEI')":"Close","('High', '^NSEI')":"High","('Low', '^NSEI')":"Low","('Open', '^NSEI')":"Open","('Volume', '^NSEI')":"Volume"})
gold = gold.rename({"('Close', 'GOLDBEES.NS')":"Close","('High', 'GOLDBEES.NS')":"High","('Low', 'GOLDBEES.NS')":"Low","('Open', 'GOLDBEES.NS')":"Open","('Volume', 'GOLDBEES.NS')":"Volume"})
usdinr = usdinr.rename({"('Close', 'USDINR=X')":"Close","('High', 'USDINR=X')":"High","('Low', 'USDINR=X')":"Low","('Open', 'USDINR=X')":"Open","('Volume', 'USDINR=X')":"Volume"})


In [4]:
bad_dates = [
    pl.datetime(2019, 12, 19),
    pl.datetime(2019, 12, 20),
]
for date in bad_dates:
    nifty = nifty.filter(pl.col("Date") != date)
    gold = gold.filter(pl.col("Date") != date)
    usdinr = usdinr.filter(pl.col("Date") != date)


In [5]:
def feature_engineering(asset):
    # base expressions
    c = pl.col("Close")
    o = pl.col("Open")
    h = pl.col("High")
    l = pl.col("Low")

    # Macd Calc
    ema_12 = c.ewm_mean(span=12,adjust=False)
    ema_26 = c.ewm_mean(span=26,adjust=False)
    macd_line = ema_12 - ema_26
    signal_line = macd_line.ewm_mean(span=9,adjust=False)


    # Rsi Calc
    delta = c.diff()
    up = delta.clip(lower_bound =0)
    down = delta.clip(upper_bound=0).abs()
    roll_up = up.ewm_mean(com=13,ignore_nulls = True)
    roll_down = down.ewm_mean(com=13,ignore_nulls = True)
    rs = roll_up/roll_down
    rsi = 100.0 -(100.0/(1.0+rs))

    # BB Calc
    bb_mean = c.rolling_mean(window_size=20)
    bb_std = c.rolling_std(window_size=20)
    bb_up = bb_mean + (2*bb_std)
    bb_low = bb_mean - (2*bb_std)

    # True Range Calc
    prev_close = c.shift(1)
    tr1 = h -l
    tr2 = (h-prev_close).abs()
    tr3 = (l-prev_close).abs()
    tr = pl.max_horizontal([tr1,tr2,tr3])
    atr = tr.ewm_mean(span=14,adjust=False)


    q = (
        asset.lazy()
        .with_columns(
            #Returns
            c.pct_change().alias("Ret_1d"),
            c.pct_change(n=3).alias("Ret_3d"),
            c.pct_change(n=5).alias("Ret_5d"),
            c.pct_change(n=20).alias("Ret_20d"),

            #MAs
            c.rolling_mean(window_size=5).alias("MA_5d"),
            c.rolling_mean(window_size=20).alias("MA_20d"),

            ((c.shift(-3)-o.shift(-1))/o.shift(-1)).alias("Forward_Return"),
            
            macd_line.alias("Macd_Line"),
            signal_line.alias("Signal_Line"),
        )
        .with_columns(
            #Volatility
            pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
            pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
            pl.col("Ret_1d").shift(1).alias("Ret_1d_Lag1"),
            pl.col("Ret_1d").shift(2).alias("Ret_1d_Lag2"),
            pl.col("Ret_1d").shift(3).alias("Ret_1d_Lag3"),

            pl.col("Ret_1d").abs().alias("Abs_Return"),
            pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
            
            (pl.col("Macd_Line")-pl.col("Signal_Line")).alias("MACD_Hist"),
            rsi.alias("RSI"),
            ((c/c.shift(10)-1)*100).alias("ROC_10"),
            ((c-bb_low)/(bb_up-bb_low)).alias("BB_Pct"),
            (atr/c).alias("ATR_Pct"),
        )
        .with_columns(
            (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
            (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
            (c/pl.col("MA_20d")).alias("Price_vs_MA20"),
            pl.when((h-l)!=0)
            .then((c-l)/(h-l))
            .otherwise(0.5)
            .alias("Close_Pos_Range"),
            ((c-o)/o).alias("Intraday_Return"),
            ((h-l)/c).rolling_mean(window_size=5).alias("Rolling_Range"),
            (pl.col("Forward_Return")>0).cast(pl.Int8).alias("Label"),
        )
        .drop(["MA_5d", "MA_20d", "Forward_Return", 'Macd_Line','Signal_Line',])
    )

    return q.collect()

In [6]:
nifty = feature_engineering(nifty)
gold = feature_engineering(gold)
usdinr = feature_engineering(usdinr)

In [7]:
exposure_features = ["Ret_1d","Ret_3d","Ret_5d","Ret_20d","Vol_5d","Vol_20d","Vol_Ratio","MA_Ratio","Close_Pos_Range","Intraday_Return","Price_vs_MA20",'MACD_Hist','RSI','ROC_10','BB_Pct',    "ATR_Pct","Ret_1d_Lag1","Ret_1d_Lag2","Ret_1d_Lag3"]
regime_features = ["Vol_5d","Vol_20d","Vol_Ratio","Abs_Return","Rolling_Abs_Return","Rolling_Range","ATR_Pct"]

In [ ]:
len(exposure_features)

3213

In [28]:
print(len(nifty))
print(len(gold))
print(len(usdinr))

3213
3224
3403


In [ ]:

nifty_dates = set(nifty["Date"].to_list())
gold_dates  = set(gold["Date"].to_list())
usd_dates   = set(usdinr["Date"].to_list())

common = nifty_dates & gold_dates & usd_dates
print(f"Common trading days: {len(common)}")
print(f"Nifty-only days:     {len(nifty_dates - common)}")
print(f"Gold-only days:      {len(gold_dates - common)}")
print(f"USDINR-only days:    {len(usd_dates - common)}")

Common trading days: 3204
Nifty-only days:     9
Gold-only days:      20
USDINR-only days:    199


In [23]:
nifty["Label"].value_counts()

Label,count
i8,u32
1,1703
null,3
0,1507


In [25]:
usdinr["Volume"].value_counts().sort(by="Volume")

Volume,count
i64,u32
0,3403


In [22]:
nifty_voume = nifty.select(["Date","Volume"]).sort("Volume")
nifty_voume.head(40)

Date,Volume
datetime[ns],i64
2013-01-02 00:00:00,0
2013-01-03 00:00:00,0
2013-01-04 00:00:00,0
2013-01-07 00:00:00,0
2013-01-08 00:00:00,0
…,…
2023-08-16 00:00:00,0
2024-02-19 00:00:00,0
2024-04-01 00:00:00,0


In [9]:
with open(CONFIG_PATH,"r") as f:
    config = yaml.safe_load(f)

config["Exposure_Features"] = exposure_features
config["Regime_Features"] = regime_features

with open(CONFIG_PATH, "w") as f:
    yaml.dump(config,f,default_flow_style=False)

In [10]:
temporal_split = [
    [pl.datetime(2013,12,31),pl.datetime(2024,1,1)],
    [pl.datetime(2024,2,29),pl.datetime(2026,1,1)]
    ]
nifty_train = nifty.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
nifty_test = nifty.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

gold_train = gold.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
gold_test = gold.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )
usdinr_train = usdinr.filter(
    (pl.col("Date")>temporal_split[0][0]) & (pl.col("Date")<temporal_split[0][1])
    )
usdinr_test = usdinr.filter(
    (pl.col("Date")>temporal_split[1][0]) & (pl.col("Date")<temporal_split[1][1])
    )

In [11]:
nifty_test.head()

Close,High,Low,Open,Volume,Date,Ret_1d,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_20d,Ret_1d_Lag1,Ret_1d_Lag2,Ret_1d_Lag3,Abs_Return,Rolling_Abs_Return,MACD_Hist,RSI,ROC_10,BB_Pct,ATR_Pct,Vol_Ratio,MA_Ratio,Price_vs_MA20,Close_Pos_Range,Intraday_Return,Rolling_Range,Label
f64,f64,f64,f64,i64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
22338.75,22353.300781,22047.75,22048.300781,351500,2024-03-01 00:00:00,0.016192,0.006325,0.005675,0.022191,0.010124,0.006649,0.001442,-0.011136,0.003449,0.016192,0.00726,-0.851248,61.71051,1.352275,0.945415,0.010646,1.522674,1.006123,1.016136,0.952379,0.013173,0.009752,1
22405.599609,22440.900391,22358.300781,22403.5,298800,2024-03-04 00:00:00,0.002993,0.020703,0.012817,0.029116,0.009691,0.006559,0.016192,0.001442,-0.011136,0.002993,0.007042,11.556427,63.095772,1.280835,0.956624,0.009807,1.477538,1.00725,1.017709,0.572628,0.000094,0.009341,1
22356.300781,22416.900391,22269.150391,22371.25,296200,2024-03-05 00:00:00,-0.0022,0.016991,0.007115,0.019467,0.009892,0.006459,0.002993,0.016192,0.001442,0.0022,0.006793,14.217602,61.333501,0.717898,0.854105,0.009399,1.531499,1.007708,1.014487,0.58985,-0.000668,0.009468,1
22474.050781,22497.199219,22224.349609,22327.5,312300,2024-03-06 00:00:00,0.005267,0.006057,0.023821,0.024785,0.006956,0.006525,-0.0022,0.002993,0.016192,0.005267,0.005619,21.258618,63.925204,1.899792,0.920108,0.009722,1.066076,1.011207,1.018574,0.91516,0.006564,0.009042,0
22493.550781,22525.650391,22430.0,22505.300781,379900,2024-03-07 00:00:00,0.000868,0.003925,0.023234,0.035712,0.007028,0.005999,0.005267,-0.0022,0.002993,0.000868,0.005504,24.357193,64.351344,1.242724,0.894399,0.008986,1.171538,1.014054,1.017669,0.664407,-0.000522,0.008073,0


In [12]:
nifty_train.write_parquet("../data/processed/train/nifty.parquet")
gold_train.write_parquet("../data/processed/train/gold.parquet")
usdinr_train.write_parquet("../data/processed/train/usdinr.parquet")
nifty_test.write_parquet("../data/processed/test/nifty.parquet")
gold_test.write_parquet("../data/processed/test/gold.parquet")
usdinr_test.write_parquet("../data/processed/test/usdinr.parquet")


In [13]:
nifty.shape

(3213, 29)